In [14]:
# 07_advanced_feature_engineering.ipynb (Final Corrected)
# -------------------------------------------------------
# Adds advanced technical & contextual features while keeping Target & Next_Close intact

import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

# =====================================================
# Configuration
# =====================================================
project_root = Path(".").resolve()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "Data" / "Processed"
enhanced_dir = processed_dir / "enhanced"
enhanced_dir.mkdir(parents=True, exist_ok=True)

print("Using processed_dir:", processed_dir)
print("Enhanced will be saved in:", enhanced_dir)

files = {
    "RELIANCE": processed_dir / "reliance_model_ready.csv",
    "TCS": processed_dir / "tcs_model_ready.csv",
    "HDFCBANK": processed_dir / "hdfcbank_model_ready.csv",
}

# =====================================================
# Helper Functions
# =====================================================
def compute_RSI(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def add_rolling_features(df, prefix, col):
    df[f"{prefix}_Return_3"] = df[col].pct_change(3)
    df[f"{prefix}_Return_5"] = df[col].pct_change(5)
    df[f"{prefix}_Return_10"] = df[col].pct_change(10)
    df[f"{prefix}_Volatility_7"] = df[col].pct_change().rolling(7).std()
    df[f"{prefix}_Volatility_14"] = df[col].pct_change().rolling(14).std()
    df[f"{prefix}_Momentum_5"] = df[col] - df[col].shift(5)
    df[f"{prefix}_Momentum_10"] = df[col] - df[col].shift(10)
    return df


def add_bollinger_bands(df, prefix, col="Close", window=20):
    rolling_mean = df[col].rolling(window).mean()
    rolling_std = df[col].rolling(window).std()
    df[f"{prefix}_BB_upper"] = rolling_mean + (2 * rolling_std)
    df[f"{prefix}_BB_lower"] = rolling_mean - (2 * rolling_std)
    df[f"{prefix}_BB_width"] = df[f"{prefix}_BB_upper"] - df[f"{prefix}_BB_lower"]
    return df


def add_crossover_flags(df, prefix):
    if f"{prefix}_MA7" in df.columns and f"{prefix}_MA21" in df.columns:
        df[f"{prefix}_MA_Crossover"] = (df[f"{prefix}_MA7"] > df[f"{prefix}_MA21"]).astype(int)
    if f"{prefix}_RSI" in df.columns:
        df[f"{prefix}_RSI_Overbought"] = (df[f"{prefix}_RSI"] > 70).astype(int)
        df[f"{prefix}_RSI_Oversold"] = (df[f"{prefix}_RSI"] < 30).astype(int)
    return df


def add_index_features(df, index_symbol="^NSEI"):
    """Adds NIFTY-based contextual features safely, even if dates don't align perfectly."""
    # --- Download NIFTY index ---
    nifty = yf.download(index_symbol, start="2018-12-31", end="2025-01-01", auto_adjust=False)
    nifty.index = pd.to_datetime(nifty.index).tz_localize(None)
    nifty = nifty[["Close"]].copy()
    nifty.columns = ["NIFTY_Close"]

    # --- Prepare df ---
    if "Date" not in df.columns:
        df["Date"] = pd.date_range(start="2018-01-01", periods=len(df), freq="D")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)

    # --- Merge on nearest available date ---
    nifty_reset = nifty.reset_index().rename(columns={"Date": "NIFTY_Date"})
    merged = pd.merge_asof(df.sort_values("Date"), nifty_reset.sort_values("NIFTY_Date"),
                           left_on="Date", right_on="NIFTY_Date", direction="backward")

    merged["NIFTY_Return"] = merged["NIFTY_Close"].pct_change()
    merged["NIFTY_MA7"] = merged["NIFTY_Close"].rolling(7).mean()
    merged["NIFTY_MA21"] = merged["NIFTY_Close"].rolling(21).mean()
    merged["NIFTY_RSI"] = compute_RSI(merged["NIFTY_Close"], 14)

    merged = merged.drop(columns=["NIFTY_Date"])
    return merged


# =====================================================
# Main Processing
# =====================================================
for ticker, filepath in files.items():
    print(f"\n=== Processing {ticker} ===")

    if not filepath.exists():
        print(f"  ⚠️ File missing: {filepath}")
        continue

    # Load dataset
    base_df = pd.read_csv(filepath)
    print(f"  Loaded base: {base_df.shape}")

    # Detect or create Date column
    if "Date" not in base_df.columns:
        first_col = base_df.columns[0]
        if pd.to_datetime(base_df[first_col], errors="coerce").notna().sum() > 0:
            base_df = base_df.rename(columns={first_col: "Date"})
        else:
            # If no usable date, create a synthetic one for continuity
            base_df["Date"] = pd.date_range(start="2018-01-01", periods=len(base_df), freq="D")
            print("  ⚠️ 'Date' column not found — synthetic dates assigned.")

    close_col = next((c for c in base_df.columns if "Close" in c), base_df.columns[1])
    df = base_df.copy()

    df = add_rolling_features(df, ticker, close_col)
    df = add_bollinger_bands(df, ticker, col=close_col)
    df = add_crossover_flags(df, ticker)
    df = add_index_features(df)

    df = df.fillna(method="ffill").fillna(method="bfill")

    # --- Clean and merge without losing all rows ---
    df = df.ffill().bfill()
    
    merged = pd.merge(base_df, df.drop(columns=["Date"], errors="ignore"),
                      left_index=True, right_index=True, how="left")
    
    # Fill remaining NaNs but keep rows intact
    merged = merged.ffill().bfill()
    merged = merged.loc[:, ~merged.columns.duplicated()].reset_index(drop=True)


    save_path = enhanced_dir / f"{ticker.lower()}_enhanced_model_ready.csv"
    merged.to_csv(save_path, index=False)
    print(f"  ✅ Saved enhanced dataset: {save_path}, shape={merged.shape}")

print("\n✅ Feature engineering completed. Enhanced datasets saved in /Data/Processed/enhanced/")


[*********************100%***********************]  1 of 1 completed

Using processed_dir: C:\JupyterProjects\Stock_ML_Project\Data\Processed
Enhanced will be saved in: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced

=== Processing RELIANCE ===
  Loaded base: (1460, 9)
  ✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\reliance_enhanced_model_ready.csv, shape=(1460, 32)

=== Processing TCS ===
  Loaded base: (1460, 9)



C:\Users\vikhy\AppData\Local\Temp\ipykernel_35784\4044030640.py:92: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  merged["NIFTY_Return"] = merged["NIFTY_Close"].pct_change()
C:\Users\vikhy\AppData\Local\Temp\ipykernel_35784\4044030640.py:133: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfill")
[*********************100%***********************]  1 of 1 completed
C:\Users\vikhy\AppData\Local\Temp\ipykernel_35784\4044030640.py:92: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not f

  ✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\tcs_enhanced_model_ready.csv, shape=(1460, 32)

=== Processing HDFCBANK ===
  Loaded base: (1460, 9)
  ✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\hdfcbank_enhanced_model_ready.csv, shape=(1460, 32)

✅ Feature engineering completed. Enhanced datasets saved in /Data/Processed/enhanced/


In [25]:
# =====================================================
# 📊 07_advanced_feature_engineering (Stable + Auto Date Fix)
# =====================================================

import pandas as pd
import numpy as np
from pathlib import Path

# =====================================================
# Configuration
# =====================================================
project_root = Path(".").resolve()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "Data" / "Processed"
enhanced_dir = processed_dir / "enhanced"
enhanced_dir.mkdir(parents=True, exist_ok=True)

print("Using processed_dir:", processed_dir)
print("Enhanced files will be saved in:", enhanced_dir)

files = {
    "RELIANCE": processed_dir / "reliance_model_ready.csv",
    "TCS": processed_dir / "tcs_model_ready.csv",
    "HDFCBANK": processed_dir / "hdfcbank_model_ready.csv",
}

# =====================================================
# 1️⃣ Load NIFTY data from stocks_features.csv
# =====================================================
nifty_source = processed_dir / "stocks_features.csv"
nifty_df = pd.read_csv(nifty_source)
print(f"✅ Loaded NIFTY source file: {nifty_source}, shape={nifty_df.shape}")

# Detect or fix date column
date_col = next((c for c in nifty_df.columns if "date" in c.lower()), None)
if not date_col:
    raise ValueError("No date column found in NIFTY source file.")
nifty_df.rename(columns={date_col: "Date"}, inplace=True)

nifty_df["Date"] = pd.to_datetime(nifty_df["Date"], errors="coerce")
nifty_df = nifty_df.sort_values("Date")

# Keep only NIFTY close values
nifty_data = nifty_df[["Date", "Close_^NSEI"]].rename(columns={"Close_^NSEI": "NIFTY_Close"})

# --- Compute derived NIFTY features ---
def compute_RSI(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period, min_periods=1).mean()
    avg_loss = loss.rolling(period, min_periods=1).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

nifty_data["NIFTY_Return"] = nifty_data["NIFTY_Close"].pct_change().fillna(0)
nifty_data["NIFTY_MA7"] = nifty_data["NIFTY_Close"].rolling(7, min_periods=1).mean()
nifty_data["NIFTY_MA21"] = nifty_data["NIFTY_Close"].rolling(21, min_periods=1).mean()
nifty_data["NIFTY_RSI"] = compute_RSI(nifty_data["NIFTY_Close"]).fillna(method="bfill")

print("\n✅ Sample NIFTY data with derived features:")
display(nifty_data.head(10))

# =====================================================
# 2️⃣ Helper functions for stock-specific features
# =====================================================
def add_rolling_features(df, prefix, col):
    df[f"{prefix}_Return_3"] = df[col].pct_change(3)
    df[f"{prefix}_Return_5"] = df[col].pct_change(5)
    df[f"{prefix}_Return_10"] = df[col].pct_change(10)
    df[f"{prefix}_Volatility_7"] = df[col].pct_change().rolling(7).std()
    df[f"{prefix}_Volatility_14"] = df[col].pct_change().rolling(14).std()
    df[f"{prefix}_Momentum_5"] = df[col] - df[col].shift(5)
    df[f"{prefix}_Momentum_10"] = df[col] - df[col].shift(10)
    return df

def add_bollinger_bands(df, prefix, col, window=20):
    rolling_mean = df[col].rolling(window, min_periods=1).mean()
    rolling_std = df[col].rolling(window, min_periods=1).std()
    df[f"{prefix}_BB_upper"] = rolling_mean + (2 * rolling_std)
    df[f"{prefix}_BB_lower"] = rolling_mean - (2 * rolling_std)
    df[f"{prefix}_BB_width"] = df[f"{prefix}_BB_upper"] - df[f"{prefix}_BB_lower"]
    return df

def add_crossover_flags(df, prefix):
    if f"{prefix}_MA7" in df.columns and f"{prefix}_MA21" in df.columns:
        df[f"{prefix}_MA_Crossover"] = (df[f"{prefix}_MA7"] > df[f"{prefix}_MA21"]).astype(int)
    if f"{prefix}_RSI" in df.columns:
        df[f"{prefix}_RSI_Overbought"] = (df[f"{prefix}_RSI"] > 70).astype(int)
        df[f"{prefix}_RSI_Oversold"] = (df[f"{prefix}_RSI"] < 30).astype(int)
    return df

# =====================================================
# 3️⃣ Main Enhancement Loop
# =====================================================
for ticker, filepath in files.items():
    print(f"\n=== Processing {ticker} ===")

    if not filepath.exists():
        print(f"⚠️ Missing file: {filepath}")
        continue

    df = pd.read_csv(filepath)

    # Detect or create Date column
    date_col = next((c for c in df.columns if "date" in c.lower()), None)
    if not date_col:
        print("⚠️ No Date column found. Creating synthetic dates...")
        df["Date"] = pd.date_range(start="2019-01-01", periods=len(df), freq="D")
    else:
        df.rename(columns={date_col: "Date"}, inplace=True)
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    close_col = next((c for c in df.columns if "Close" in c), df.columns[1])

    # Add stock-specific features
    df = add_rolling_features(df, ticker, close_col)
    df = add_bollinger_bands(df, ticker, col=close_col)
    df = add_crossover_flags(df, ticker)

    # Merge NIFTY features directly
    df = pd.merge_asof(
        df.sort_values("Date"),
        nifty_data.sort_values("Date"),
        on="Date",
        direction="backward"
    )

    df = df.ffill().bfill()
    save_path = enhanced_dir / f"{ticker.lower()}_enhanced_model_ready.csv"
    df.to_csv(save_path, index=False)
    print(f"✅ Saved enhanced dataset: {save_path}, shape={df.shape}")

print("\n🎯 All enhanced datasets created successfully with valid NIFTY features inline.")


Using processed_dir: C:\JupyterProjects\Stock_ML_Project\Data\Processed
Enhanced files will be saved in: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced
✅ Loaded NIFTY source file: C:\JupyterProjects\Stock_ML_Project\Data\Processed\stocks_features.csv, shape=(1461, 39)

✅ Sample NIFTY data with derived features:


C:\Users\vikhy\AppData\Local\Temp\ipykernel_35784\1212743058.py:61: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nifty_data["NIFTY_RSI"] = compute_RSI(nifty_data["NIFTY_Close"]).fillna(method="bfill")


,Date,NIFTY_Close,NIFTY_Return,NIFTY_MA7,NIFTY_MA21,NIFTY_RSI
0,2019-01-29,10652.200195,0.000000,10652.200195,10652.200195,0.000000
1,2019-01-30,10651.799805,-0.000038,10652.000000,10652.000000,0.000000
2,2019-01-31,10830.950195,0.016819,10711.650065,10711.650065,99.777004
3,2019-02-01,10893.650391,0.005789,10757.150146,10757.150146,99.834721
4,2019-02-04,10912.250000,0.001707,10788.170117,10788.170117,99.846506
5,2019-02-05,10934.349609,0.002025,10812.533366,10812.533366,99.858494
6,2019-02-06,11062.450195,0.011715,10848.235770,10848.235770,99.902593
7,2019-02-07,11069.400391,0.000628,10907.835798,10875.881348,99.904213
8,2019-02-08,10943.599609,-0.011365,10949.521484,10883.405599,76.792798
9,2019-02-11,10888.799805,-0.005007,10957.785714,10883.945020,69.762696



=== Processing RELIANCE ===
⚠️ No Date column found. Creating synthetic dates...
✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\reliance_enhanced_model_ready.csv, shape=(1460, 25)

=== Processing TCS ===
⚠️ No Date column found. Creating synthetic dates...
✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\tcs_enhanced_model_ready.csv, shape=(1460, 25)

=== Processing HDFCBANK ===
⚠️ No Date column found. Creating synthetic dates...
✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\hdfcbank_enhanced_model_ready.csv, shape=(1460, 25)

🎯 All enhanced datasets created successfully with valid NIFTY features inline.


In [23]:
import pandas as pd
from pathlib import Path

# Load the combined stock + NIFTY features file
file_path = Path("C:/JupyterProjects/Stock_ML_Project/Data/Processed/stocks_features.csv")
df = pd.read_csv(file_path)

# Show the first 5 rows and dataset shape
print("✅ File loaded successfully:", file_path)
display(df.head())
print("\n📊 Dataset shape:", df.shape)


✅ File loaded successfully: C:\JupyterProjects\Stock_ML_Project\Data\Processed\stocks_features.csv


,Date,Close_HDFCBANK.NS,Close_RELIANCE.NS,Close_TCS.NS,Close_^NSEI,High_HDFCBANK.NS,High_RELIANCE.NS,High_TCS.NS,High_^NSEI,Low_HDFCBANK.NS,...,TCS.NS_MA21,TCS.NS_EMA21,TCS.NS_STD21,TCS.NS_RSI,HDFCBANK.NS_Return,HDFCBANK.NS_MA7,HDFCBANK.NS_MA21,HDFCBANK.NS_EMA21,HDFCBANK.NS_STD21,HDFCBANK.NS_RSI
0,2019-01-29,483.142303,538.356567,1682.212524,10652.200195,489.106420,547.694960,1687.557637,10690.349609,482.062161,...,1606.211577,1615.142729,30.753741,65.781201,-0.012739,494.084390,496.479430,496.250070,4.896800,32.222165
1,2019-01-30,477.765167,531.708435,1681.067139,10651.799805,483.060114,544.737715,1687.939352,10710.200195,476.191961,...,1609.551519,1621.135857,34.830087,65.327356,-0.011130,490.278809,495.211461,494.569625,6.055647,29.624470
2,2019-01-31,488.390259,545.693909,1708.810547,10830.950195,490.691395,546.827850,1713.604163,10838.049805,474.783108,...,1613.386132,1629.106283,40.899493,78.971761,0.022239,488.403687,494.668605,494.007864,6.135113,42.834774
3,2019-02-01,490.855743,555.832520,1722.258179,10893.650391,497.089925,558.078198,1726.330655,10983.450195,487.603678,...,1618.802473,1637.574637,47.259229,88.099281,0.005048,487.902200,494.429879,493.721308,6.183415,47.490990
4,2019-02-04,494.248749,574.042419,1739.481201,10912.250000,496.150722,576.732769,1743.892964,10927.900391,489.024256,...,1625.970203,1646.838870,53.508179,86.508463,0.006912,487.991093,494.289552,493.769257,6.150864,46.090219



📊 Dataset shape: (1461, 39)
